# BLOCK-T438 - TMA Dynamic Tests

In [ ]:
# %load_ext lab_black
import warnings
import numpy as np
import os

from lsst.ts.observing import ObservingBlock, ObservingScript

In [ ]:
name = "BLOCK-T438"
program = "BLOCK-T438"
reason = "BLOCK-T438"
constraints = []
scripts = []

try:
    output_folder = (
        os.environ["TS_CONFIG_OCS_DIR"] + "/Scheduler/observing_blocks_maintel"
    )
except KeyError:
    warnings.warn(
        "The environment variable 'TS_CONFIG_OCS_DIR' is not set. Using default folder 'output_blocks'."
    )
    output_folder = "output_blocks"

In [ ]:
def build_configuration_schema(block_number, properties):
    """
    Builds a configuration schema string for a given BLOCK and configurable
    properties.

    Parameters
    ----------
    block_number :
        The BLOCK number to include in the title and description.
    properties : dict
        A dictionary where each key is a property name, and each value is a
        dictionary with keys 'description', 'type', and optionally 'default'.

    Returns
    -------
        A formatted configuration schema string.
    """

    # Define the base schema with the BLOCK number
    configuration_schema = (
        "$schema: http://json-schema.org/draft-07/schema#\n"
        f"title: BLOCK-{block_number} configuration\n"
        f"description: Configuration for BLOCK-{block_number}.\n"
        "type: object\n"
        "properties:\n"
    )

    # Add each property to the schema
    for prop_name, prop_details in properties.items():
        configuration_schema += f"  {prop_name}:\n"
        configuration_schema += f'    description: {prop_details["description"]}\n'
        # Handle single type vs list of types
        types = prop_details["type"]
        if isinstance(types, list):
            configuration_schema += "    anyOf:\n"
            for t in types:
                configuration_schema += f"      - type: {t}\n"
        else:
            configuration_schema += f"    type: {types}\n"
        # Add default if present
        if "default" in prop_details:
            default_value = prop_details["default"]
            # Quote string defaults
            if isinstance(default_value, str) and types == "string":
                default_value = f'"{default_value}"'
            configuration_schema += f"    default: {default_value}\n"

    return configuration_schema

Define the configurable properties that we will use in the configuration schema

In [ ]:
# Defining configurable properties
properties = {
    "grid_az": {
        "description": "Grid of azimuth positions to start short and long slews",
        "type": ["number", "array"],
        "default": -75,
    },
    "grid_el": {
        "description": "Grid of elevation positions to start short and long slews",
        "type": ["number", "array"],
        "default": 70,
    },
    "direction": {
        "description": "Direction of the first slew ('forward' or 'backward')",
        "type": "string",
        "default": "forward",
    },
    "pause_for": {
        "description": "Pause duration between movements in seconds",
        "type": "number",
        "default": 30,
    },
    "move_timeout": {
        "description": "Timeout for each move command",
        "type": "number",
        "default": 120,
    },
    "ignore": {
        "description": "Name of the CSCs we want to ignore",
        "type": "array",
        "default": ["mtaos", "mtdome", "mtdometrajectory"],
    },
}

block_number = name.split("-")[-1]
configuration_schema = build_configuration_schema(block_number, properties)
print(configuration_schema)

In [ ]:
# Define the ObservingBlock
slews = ObservingScript(
    name="maintel/tma/short_long_slews.py",
    standard=False,
    parameters=dict(
        grid_az="$grid_az",
        grid_el="$grid_el",
        direction="$direction",
        pause_for="$pause_for",
        move_timeout="$move_timeout",
        ignore="$ignore",
    ),
)

scripts.append(slews)

In [ ]:
block = ObservingBlock(
    name=name,
    program=program,
    configuration_schema=configuration_schema,
    scripts=scripts,
)

In [ ]:
# block.model_dump_json(indent=2)

os.makedirs(output_folder, exist_ok=True)
output_path = f"{output_folder}/{name}.json"

with open(output_path, "w") as file:
    file.write(block.model_dump_json(indent=2))